# Machine Learning Models

### Imports and Set up

In [24]:
import numpy as np
import pandas as pd
import pickle
import time
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

### Load data

In [25]:
with open('group_5_dataset.pkl', 'rb') as f:
    data = pickle.load(f)

# data format from pickle file:
# 'X_train': (n_train, T, 4),
# 'X_val':   (n_val,   T, 4),
# 'X_test':  (n_test,  T, 4),
# 'y_train': (n_train, 2),   # [R, C]
# 'y_val':   (n_val,   2),
# 'y_test':  (n_test,  2),
# optional: 'mu', 'sigma', 'indices', 'target_is_log'
target_is_log = data.get('target_is_log', False)

# Keep time-series waveforms and scaling stats for Gauss-Newton
X_test_ts = data['X_test']                   # (n_test, T, 4)
mu    = data.get('mu',    None)               # (1, 1, 4) channel means
sigma = data.get('sigma', None)               # (1, 1, 4) channel stds

# flatten out the data from pickle file
X_train = data['X_train'].reshape(data['X_train'].shape[0], -1)  # (n_train, 2000)
X_val = data['X_val'].reshape(data['X_val'].shape[0], -1)        # (n_val, 2000)
X_test = data['X_test'].reshape(data['X_test'].shape[0], -1)     # (n_test, 2000)
y_train = data['y_train']  # (n_train, 2)
y_val = data['y_val']       # (n_val, 2)
y_test = data['y_test']    # (n_test, 2)

### Data preview 

Each sample has **2000 features**: 500 time steps × 4 channels (V1, V2, V3, I_E). At each time step the 4 values are [V1, V2, V3, I_E]. Features 0–3 = first time step, 4–7 = second, etc. Below: first 8 (2 time steps) + targets R (Ω), C (F).

In [26]:
# First 20 rows: first 8 features = first 2 time steps (each step: V1, V2, V3, I_E) + targets
n_display = 20
n_feat = X_train.shape[1]
channel_names = ["V1", "V2", "V3", "I_E"]
n_show = min(8, n_feat)
cols = [f"t{i//4}_{channel_names[i%4]}" for i in range(n_show)]
df_feat = pd.DataFrame(X_train[:n_display, :n_show], columns=cols)
df_targets = pd.DataFrame(y_train[:n_display], columns=["R (ohm)", "C (F)"])
df_preview = pd.concat([df_feat, df_targets], axis=1)
print("First 20 samples (first 2 time steps: t0_V1..t0_I_E, t1_V1..t1_I_E) + targets:")
display(df_preview)
print(f"\nTotal features per sample: {X_train.shape[1]} (500 steps × 4 channels). Target ranges — R: [{y_train[:, 0].min():.2f}, {y_train[:, 0].max():.2f}] Ω,  C: [{y_train[:, 1].min():.2e}, {y_train[:, 1].max():.2e}] F")

First 20 samples (first 2 time steps: t0_V1..t0_I_E, t1_V1..t1_I_E) + targets:


,t0_V1,t0_V2,t0_V3,t0_I_E,t1_V1,t1_V2,t1_V3,t1_I_E,R (ohm),C (F)
0,-0.003269,0.031026,-0.925145,0.394545,0.043481,0.132591,-0.896686,0.409420,6.738612,-15.257172
1,-0.016245,0.010331,-0.940233,0.395156,0.014580,0.069170,-0.912831,0.392038,7.551296,-14.064571
2,0.043841,0.035361,-0.903696,0.394957,0.076210,0.051829,-0.909713,0.395193,7.246409,-13.995175
3,0.007304,0.084272,-0.958441,0.400161,0.063790,0.045778,-0.907941,0.396464,7.217771,-14.025276
4,0.005910,0.070632,-0.902679,0.402762,0.047664,0.054034,-0.927223,0.410611,7.201727,-12.926674
5,0.010445,0.041617,-0.925286,0.397179,0.051957,0.085442,-0.895612,0.385305,7.567712,-12.417433
6,0.012905,0.044780,-0.905386,0.386303,0.057084,0.065452,-0.896447,0.385115,7.458938,-13.243097
7,-0.002115,0.024820,-0.893830,0.398416,0.087966,0.125739,-0.889481,0.378612,6.989643,-12.870809
8,-0.023870,0.039627,-0.905127,0.370614,0.011818,0.052285,-0.914150,0.397832,6.003136,-12.333643
9,0.016910,0.027648,-0.880315,0.407627,0.052752,0.096419,-0.910042,0.383176,7.151654,-12.345269



Total features per sample: 2000 (500 steps × 4 channels). Target ranges — R: [0.24, 7.82] Ω,  C: [-1.61e+01, -1.22e+01] F


### Train / validation / test split

In [27]:
n_train, n_val, n_test = len(X_train), len(X_val), len(X_test)
n_total = n_train + n_val + n_test
split_df = pd.DataFrame({
    "set": ["train", "validation", "test"],
    "samples": [n_train, n_val, n_test],
    "fraction (%)": [100 * n_train / n_total, 100 * n_val / n_total, 100 * n_test / n_total],
})
print("Split (from pickle):")
display(split_df)
print(f"Total samples: {n_total}  |  Features per sample: {X_train.shape[1]}  |  Targets: R, C")

Split (from pickle):


,set,samples,fraction (%)
0,train,1400,70.0
1,validation,300,15.0
2,test,300,15.0


Total samples: 2000  |  Features per sample: 2000  |  Targets: R, C


## 1. Linear regression

We use **Ridge regression** (linear model with L2 regularization) and tune the regularization strength `alpha` on the validation set. For each configuration we record **MAE**, **RMSE**, and **MAPE** separately for **R** and **C**, as well as training time and data quantity.

Because **C** is orders of magnitude smaller than **R**, we select the best configuration using the lowest average validation MAPE across both targets:

In [28]:
target_is_log = globals().get('target_is_log', False)
y_train_phys = np.exp(y_train) if target_is_log else y_train
y_val_phys   = np.exp(y_val)   if target_is_log else y_val
y_test_phys  = np.exp(y_test)  if target_is_log else y_test

def to_phys(y_pred, is_log):
    return np.exp(y_pred) if is_log else y_pred

def mape_r_c(y_true, y_pred, eps=1e-12):
    denom_r = np.maximum(np.abs(y_true[:, 0]), eps)
    denom_c = np.maximum(np.abs(y_true[:, 1]), eps)
    return (np.mean(np.abs(y_true[:, 0] - y_pred[:, 0]) / denom_r) * 100,
            np.mean(np.abs(y_true[:, 1] - y_pred[:, 1]) / denom_c) * 100)

def metrics_r_c(y_true, y_pred):
    mae_r  = mean_absolute_error(y_true[:, 0], y_pred[:, 0])
    mae_c  = mean_absolute_error(y_true[:, 1], y_pred[:, 1])
    rmse_r = np.sqrt(mean_squared_error(y_true[:, 0], y_pred[:, 0]))
    rmse_c = np.sqrt(mean_squared_error(y_true[:, 1], y_pred[:, 1]))
    mr, mc = mape_r_c(y_true, y_pred)
    return mae_r, mae_c, rmse_r, rmse_c, mr, mc

def _fmt_df(df):
    c_cols    = [c for c in df.columns if '_mae_c'  in c or '_rmse_c' in c]
    r_cols    = [c for c in df.columns if '_mae_r'  in c or '_rmse_r' in c]
    mape_cols = [c for c in df.columns if '_mape_'  in c]
    t_cols    = ['train_time_s'] if 'train_time_s' in df.columns else []
    fmt = {col: "{:.4e}" for col in c_cols}
    fmt.update({col: "{:.4f}" for col in r_cols})
    fmt.update({col: "{:.2f}%" for col in mape_cols})
    fmt.update({col: "{:.4f}" for col in t_cols})
    return df.style.format(fmt)

n_train, n_val, n_test = len(X_train), len(X_val), len(X_test)
alpha_list = [1e-4, 1e-3, 1e-2, 0.1, 1.0, 10.0, 100.0]
lr_results = []
best_val_mape_avg = float('inf')
lr_model      = None
train_time_lr = None
best_lr_params = {}

for alpha in alpha_list:
    model = Ridge(alpha=alpha, random_state=42)
    t0 = time.perf_counter()
    model.fit(X_train, y_train)
    t_elapsed = time.perf_counter() - t0
    y_train_pred = to_phys(model.predict(X_train), target_is_log)
    y_val_pred   = to_phys(model.predict(X_val),   target_is_log)
    y_test_pred  = to_phys(model.predict(X_test),  target_is_log)
    tr = metrics_r_c(y_train_phys, y_train_pred)
    va = metrics_r_c(y_val_phys,   y_val_pred)
    te = metrics_r_c(y_test_phys,  y_test_pred)
    val_mape_avg = (va[4] + va[5]) / 2
    lr_results.append({
        'alpha': alpha,
        'train_time_s': t_elapsed,
        'train_mae_r':  tr[0], 'train_mae_c':  tr[1],
        'train_rmse_r': tr[2], 'train_rmse_c': tr[3],
        'train_mape_r': tr[4], 'train_mape_c': tr[5],
        'val_mae_r':    va[0], 'val_mae_c':    va[1],
        'val_rmse_r':   va[2], 'val_rmse_c':   va[3],
        'val_mape_r':   va[4], 'val_mape_c':   va[5],
        'test_mae_r':   te[0], 'test_mae_c':   te[1],
        'test_rmse_r':  te[2], 'test_rmse_c':  te[3],
        'test_mape_r':  te[4], 'test_mape_c':  te[5],
    })
    if val_mape_avg < best_val_mape_avg:
        best_val_mape_avg = val_mape_avg
        lr_model      = model
        train_time_lr = t_elapsed
        best_lr_params = {'alpha': alpha}

lr_experiments_df = pd.DataFrame(lr_results)
print("Linear Regression (Ridge) — hyperparameter tuning")
print(f"Data: n_train={n_train}, n_val={n_val}, n_test={n_test}")
print("Selection criterion: lowest avg validation MAPE across R and C.")
display(_fmt_df(lr_experiments_df))
print(f"\nBest config (by validation MAPE): {best_lr_params}")
print("Justification: Chosen alpha minimizes average validation MAPE across R and C.")
print("\nBest model — Val and Test metrics:")
y_val_pred  = to_phys(lr_model.predict(X_val),  target_is_log)
y_test_pred = to_phys(lr_model.predict(X_test), target_is_log)
va = metrics_r_c(y_val_phys,  y_val_pred)
te = metrics_r_c(y_test_phys, y_test_pred)
print(f"  Train time: {train_time_lr:.4f} s")
print(f"  Val:  R mae: {va[0]:.4f}  C mae: {va[1]:.4e}  R rmse: {va[2]:.4f}  C rmse: {va[3]:.4e}  R mape: {va[4]:.2f}%  C mape: {va[5]:.2f}%")
print(f"  Test: R mae: {te[0]:.4f}  C mae: {te[1]:.4e}  R rmse: {te[2]:.4f}  C rmse: {te[3]:.4e}  R mape: {te[4]:.2f}%  C mape: {te[5]:.2f}%")

Linear Regression (Ridge) — hyperparameter tuning
Data: n_train=1400, n_val=300, n_test=300
Selection criterion: lowest avg validation MAPE across R and C.


,alpha,train_time_s,train_mae_r,train_mae_c,train_rmse_r,train_rmse_c,train_mape_r,train_mape_c,val_mae_r,val_mae_c,val_rmse_r,val_rmse_c,val_mape_r,val_mape_c,test_mae_r,test_mae_c,test_rmse_r,test_rmse_c,test_mape_r,test_mape_c
0,0.000100,0.1374,0.0584,7.4584e-11,0.0859,1.3292e-10,0.00%,0.01%,142.4504,2.3959e-07,200.2643,5.0750e-07,14.22%,18.87%,147.5221,2.1146e-07,209.2876,4.0022e-07,14.02%,17.36%
1,0.001000,0.0755,0.5712,7.3065e-10,0.8394,1.3022e-09,0.04%,0.06%,141.3274,2.3803e-07,198.5824,5.0574e-07,14.10%,18.73%,146.3168,2.0993e-07,207.8456,3.9828e-07,13.91%,17.24%
2,0.010000,0.1245,4.7319,6.1502e-09,6.9610,1.0947e-08,0.36%,0.50%,132.1962,2.2555e-07,185.2685,4.9137e-07,13.17%,17.71%,136.8242,1.9811e-07,196.5346,3.8503e-07,13.00%,16.32%
3,0.100000,0.0940,21.0839,2.8522e-08,31.2931,5.0427e-08,1.62%,2.41%,103.8737,1.8215e-07,142.0263,4.3138e-07,10.37%,14.72%,109.0920,1.6293e-07,160.6166,3.6124e-07,10.38%,13.72%
4,1.000000,0.1180,51.4473,6.6989e-08,78.1770,1.3001e-07,4.22%,6.41%,79.1693,1.3785e-07,111.1425,3.5050e-07,7.82%,13.15%,94.1966,1.4024e-07,137.9576,3.5513e-07,8.74%,12.49%
5,10.000000,0.1689,105.2429,1.3058e-07,169.7442,3.5279e-07,9.85%,13.43%,108.5209,1.2800e-07,163.4219,3.1104e-07,10.27%,15.70%,115.0336,1.9270e-07,182.5126,5.2894e-07,11.14%,15.83%
6,100.000000,0.0837,169.1017,2.2331e-07,264.0954,5.3193e-07,16.73%,22.85%,162.2094,1.9748e-07,243.7796,4.6276e-07,15.70%,23.05%,165.6914,2.7144e-07,253.9192,7.2568e-07,16.44%,22.72%



Best config (by validation MAPE): {'alpha': 1.0}
Justification: Chosen alpha minimizes average validation MAPE across R and C.

Best model — Val and Test metrics:
  Train time: 0.1180 s
  Val:  R mae: 79.1693  C mae: 1.3785e-07  R rmse: 111.1425  C rmse: 3.5050e-07  R mape: 7.82%  C mape: 13.15%
  Test: R mae: 94.1966  C mae: 1.4024e-07  R rmse: 137.9576  C rmse: 3.5513e-07  R mape: 8.74%  C mape: 12.49%


## 2. Random forest

We tune `n_estimators` and `max_depth` on the validation set. For each configuration we record **MAE**, **RMSE**, and **MAPE** separately for **R** and **C**, as well as training time and data quantity.

The best configuration is selected using the **lowest average validation MAPE** across both targets.

In [29]:
target_is_log = globals().get('target_is_log', False)
y_train_phys = np.exp(y_train) if target_is_log else y_train
y_val_phys   = np.exp(y_val)   if target_is_log else y_val
y_test_phys  = np.exp(y_test)  if target_is_log else y_test

def to_phys(y_pred, is_log):
    return np.exp(y_pred) if is_log else y_pred

def mape_r_c(y_true, y_pred, eps=1e-12):
    denom_r = np.maximum(np.abs(y_true[:, 0]), eps)
    denom_c = np.maximum(np.abs(y_true[:, 1]), eps)
    return (np.mean(np.abs(y_true[:, 0] - y_pred[:, 0]) / denom_r) * 100,
            np.mean(np.abs(y_true[:, 1] - y_pred[:, 1]) / denom_c) * 100)

def metrics_r_c(y_true, y_pred):
    mae_r  = mean_absolute_error(y_true[:, 0], y_pred[:, 0])
    mae_c  = mean_absolute_error(y_true[:, 1], y_pred[:, 1])
    rmse_r = np.sqrt(mean_squared_error(y_true[:, 0], y_pred[:, 0]))
    rmse_c = np.sqrt(mean_squared_error(y_true[:, 1], y_pred[:, 1]))
    mr, mc = mape_r_c(y_true, y_pred)
    return mae_r, mae_c, rmse_r, rmse_c, mr, mc

def _fmt_df(df):
    c_cols    = [c for c in df.columns if '_mae_c'  in c or '_rmse_c' in c]
    r_cols    = [c for c in df.columns if '_mae_r'  in c or '_rmse_r' in c]
    mape_cols = [c for c in df.columns if '_mape_'  in c]
    t_cols    = ['train_time_s'] if 'train_time_s' in df.columns else []
    fmt = {col: "{:.4e}" for col in c_cols}
    fmt.update({col: "{:.4f}" for col in r_cols})
    fmt.update({col: "{:.2f}%" for col in mape_cols})
    fmt.update({col: "{:.4f}" for col in t_cols})
    return df.style.format(fmt)

n_train, n_val, n_test = len(X_train), len(X_val), len(X_test)
n_estimators_list = [10, 50, 100, 200]
max_depth_list    = [5, 10, 20, None]
rf_results = []
best_model        = None
best_val_mape_avg = float('inf')
best_rf_params    = {}
train_time_rf     = None

for n_estimators in n_estimators_list:
    for max_depth in max_depth_list:
        model = RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        t0 = time.perf_counter()
        model.fit(X_train, y_train)
        t_elapsed = time.perf_counter() - t0
        y_train_pred = to_phys(model.predict(X_train), target_is_log)
        y_val_pred   = to_phys(model.predict(X_val),   target_is_log)
        y_test_pred  = to_phys(model.predict(X_test),  target_is_log)
        tr = metrics_r_c(y_train_phys, y_train_pred)
        va = metrics_r_c(y_val_phys,   y_val_pred)
        te = metrics_r_c(y_test_phys,  y_test_pred)
        val_mape_avg = (va[4] + va[5]) / 2
        rf_results.append({
            'n_estimators': n_estimators,
            'max_depth':    str(max_depth),
            'train_time_s': t_elapsed,
            'train_mae_r':  tr[0], 'train_mae_c':  tr[1],
            'train_rmse_r': tr[2], 'train_rmse_c': tr[3],
            'train_mape_r': tr[4], 'train_mape_c': tr[5],
            'val_mae_r':    va[0], 'val_mae_c':    va[1],
            'val_rmse_r':   va[2], 'val_rmse_c':   va[3],
            'val_mape_r':   va[4], 'val_mape_c':   va[5],
            'test_mae_r':   te[0], 'test_mae_c':   te[1],
            'test_rmse_r':  te[2], 'test_rmse_c':  te[3],
            'test_mape_r':  te[4], 'test_mape_c':  te[5],
        })
        if val_mape_avg < best_val_mape_avg:
            best_val_mape_avg = val_mape_avg
            best_model     = model
            best_rf_params  = {'n_estimators': n_estimators, 'max_depth': max_depth}
            train_time_rf   = t_elapsed

rf_experiments_df = pd.DataFrame(rf_results)
print("Random Forest — hyperparameter tuning")
print(f"Data: n_train={n_train}, n_val={n_val}, n_test={n_test}")
print("Selection criterion: lowest avg validation MAPE across R and C.")
display(_fmt_df(rf_experiments_df))
print(f"\nBest config (by validation MAPE): {best_rf_params}")
print("Justification: Chosen n_estimators and max_depth minimize average validation MAPE across R and C.")
print("\nBest model — Val and Test metrics:")
y_val_pred  = to_phys(best_model.predict(X_val),  target_is_log)
y_test_pred = to_phys(best_model.predict(X_test), target_is_log)
va = metrics_r_c(y_val_phys,  y_val_pred)
te = metrics_r_c(y_test_phys, y_test_pred)
print(f"  Train time: {train_time_rf:.4f} s")
print(f"  Val:  R mae: {va[0]:.4f}  C mae: {va[1]:.4e}  R rmse: {va[2]:.4f}  C rmse: {va[3]:.4e}  R mape: {va[4]:.2f}%  C mape: {va[5]:.2f}%")
print(f"  Test: R mae: {te[0]:.4f}  C mae: {te[1]:.4e}  R rmse: {te[2]:.4f}  C rmse: {te[3]:.4e}  R mape: {te[4]:.2f}%  C mape: {te[5]:.2f}%")

Random Forest — hyperparameter tuning
Data: n_train=1400, n_val=300, n_test=300
Selection criterion: lowest avg validation MAPE across R and C.


,n_estimators,max_depth,train_time_s,train_mae_r,train_mae_c,train_rmse_r,train_rmse_c,train_mape_r,train_mape_c,val_mae_r,val_mae_c,val_rmse_r,val_rmse_c,val_mape_r,val_mape_c,test_mae_r,test_mae_c,test_rmse_r,test_rmse_c,test_mape_r,test_mape_c
0,10,5,6.1996,121.2879,1.7103e-07,172.8115,3.4294e-07,10.35%,12.96%,119.2595,1.5558e-07,174.8219,3.4677e-07,9.76%,14.65%,119.1751,2.2664e-07,168.0037,4.6848e-07,10.90%,16.02%
1,10,10,10.3393,20.6580,4.2522e-08,30.1161,1.2840e-07,2.29%,4.45%,40.2102,7.6947e-08,59.8381,2.6153e-07,4.04%,9.43%,36.0778,9.7984e-08,49.8546,3.1755e-07,4.55%,9.72%
2,10,20,9.7575,19.2408,3.9193e-08,27.7263,1.2632e-07,2.20%,4.20%,39.9623,7.3943e-08,56.5035,2.6250e-07,4.22%,8.87%,34.8265,1.1048e-07,48.4566,3.3229e-07,4.51%,9.76%
3,10,None,9.7176,19.2408,3.9193e-08,27.7263,1.2632e-07,2.20%,4.20%,39.9623,7.3943e-08,56.5035,2.6250e-07,4.22%,8.87%,34.8265,1.1048e-07,48.4566,3.3229e-07,4.51%,9.76%
4,50,5,28.8361,122.5490,1.5612e-07,175.2178,3.2007e-07,10.09%,11.73%,121.6188,1.4105e-07,178.0672,3.3009e-07,9.86%,13.17%,120.9190,2.1273e-07,170.8014,4.3893e-07,10.97%,14.92%
5,50,10,46.1316,14.0689,3.2280e-08,20.9943,1.0826e-07,1.44%,3.42%,28.1196,6.3566e-08,44.6819,2.3812e-07,2.82%,8.14%,24.1463,8.9313e-08,35.2192,2.9112e-07,3.12%,8.69%
6,50,20,48.4983,11.8543,3.0199e-08,17.5453,1.0790e-07,1.29%,3.28%,28.7983,6.7334e-08,45.0740,2.3883e-07,2.95%,8.41%,24.3591,9.1771e-08,35.9532,2.9698e-07,3.09%,8.69%
7,50,None,49.2097,11.8543,3.0199e-08,17.5453,1.0790e-07,1.29%,3.28%,28.7983,6.7334e-08,45.0740,2.3883e-07,2.95%,8.41%,24.3591,9.1771e-08,35.9532,2.9698e-07,3.09%,8.69%
8,100,5,58.9348,120.0820,1.5789e-07,172.9084,3.2371e-07,9.82%,11.75%,120.2650,1.4148e-07,176.2725,3.3259e-07,9.74%,13.03%,120.5021,2.1238e-07,169.9987,4.4472e-07,10.83%,14.81%
9,100,10,95.0063,12.4990,2.9984e-08,19.3087,1.0252e-07,1.21%,3.29%,25.7070,6.2352e-08,42.5078,2.3796e-07,2.50%,7.83%,21.8139,8.8200e-08,33.0390,2.9549e-07,2.70%,8.52%



Best config (by validation MAPE): {'n_estimators': 200, 'max_depth': 10}
Justification: Chosen n_estimators and max_depth minimize average validation MAPE across R and C.

Best model — Val and Test metrics:
  Train time: 185.5848 s
  Val:  R mae: 24.0304  C mae: 6.1711e-08  R rmse: 41.1482  C rmse: 2.3551e-07  R mape: 2.30%  C mape: 8.02%
  Test: R mae: 20.9666  C mae: 8.6104e-08  R rmse: 31.9406  C rmse: 2.8657e-07  R mape: 2.73%  C mape: 8.57%


## 3. MLP

We tune `hidden_layer_sizes` and L2 regularization `alpha` on the validation set. For each configuration we record **MAE**, **RMSE**, and **MAPE** separately for **R** and **C**, as well as training time and data quantity.

The best configuration is selected using the **lowest average validation MAPE** across both targets.


In [30]:
target_is_log = globals().get('target_is_log', False)
y_train_phys = np.exp(y_train) if target_is_log else y_train
y_val_phys   = np.exp(y_val)   if target_is_log else y_val
y_test_phys  = np.exp(y_test)  if target_is_log else y_test

def to_phys(y_pred, is_log):
    return np.exp(y_pred) if is_log else y_pred

def mape_r_c(y_true, y_pred, eps=1e-12):
    denom_r = np.maximum(np.abs(y_true[:, 0]), eps)
    denom_c = np.maximum(np.abs(y_true[:, 1]), eps)
    return (np.mean(np.abs(y_true[:, 0] - y_pred[:, 0]) / denom_r) * 100,
            np.mean(np.abs(y_true[:, 1] - y_pred[:, 1]) / denom_c) * 100)

def metrics_r_c(y_true, y_pred):
    mae_r  = mean_absolute_error(y_true[:, 0], y_pred[:, 0])
    mae_c  = mean_absolute_error(y_true[:, 1], y_pred[:, 1])
    rmse_r = np.sqrt(mean_squared_error(y_true[:, 0], y_pred[:, 0]))
    rmse_c = np.sqrt(mean_squared_error(y_true[:, 1], y_pred[:, 1]))
    mr, mc = mape_r_c(y_true, y_pred)
    return mae_r, mae_c, rmse_r, rmse_c, mr, mc

def _fmt_df(df):
    c_cols    = [c for c in df.columns if '_mae_c'  in c or '_rmse_c' in c]
    r_cols    = [c for c in df.columns if '_mae_r'  in c or '_rmse_r' in c]
    mape_cols = [c for c in df.columns if '_mape_'  in c]
    t_cols    = ['train_time_s'] if 'train_time_s' in df.columns else []
    fmt = {col: "{:.4e}" for col in c_cols}
    fmt.update({col: "{:.4f}" for col in r_cols})
    fmt.update({col: "{:.2f}%" for col in mape_cols})
    fmt.update({col: "{:.4f}" for col in t_cols})
    return df.style.format(fmt)

n_train, n_val, n_test = len(X_train), len(X_val), len(X_test)
hidden_sizes_list = [(64, 32), (128, 64), (256, 128, 64)]
alpha_list        = [1e-4, 1e-3, 1e-2]
mlp_results = []
mlp_model         = None
best_val_mape_avg = float('inf')
best_mlp_params   = {}
train_time_mlp    = None

for hidden in hidden_sizes_list:
    for alpha in alpha_list:
        model = MLPRegressor(
            hidden_layer_sizes=hidden,
            activation='relu',
            solver='adam',
            alpha=alpha,
            max_iter=500,
            random_state=42,
            early_stopping=True,
            validation_fraction=0.1,
        )
        t0 = time.perf_counter()
        model.fit(X_train, y_train)
        t_elapsed = time.perf_counter() - t0
        y_train_pred = to_phys(model.predict(X_train), target_is_log)
        y_val_pred   = to_phys(model.predict(X_val),   target_is_log)
        y_test_pred  = to_phys(model.predict(X_test),  target_is_log)
        tr = metrics_r_c(y_train_phys, y_train_pred)
        va = metrics_r_c(y_val_phys,   y_val_pred)
        te = metrics_r_c(y_test_phys,  y_test_pred)
        val_mape_avg = (va[4] + va[5]) / 2
        mlp_results.append({
            'hidden_layer_sizes': str(hidden),
            'alpha':        alpha,
            'train_time_s': t_elapsed,
            'train_mae_r':  tr[0], 'train_mae_c':  tr[1],
            'train_rmse_r': tr[2], 'train_rmse_c': tr[3],
            'train_mape_r': tr[4], 'train_mape_c': tr[5],
            'val_mae_r':    va[0], 'val_mae_c':    va[1],
            'val_rmse_r':   va[2], 'val_rmse_c':   va[3],
            'val_mape_r':   va[4], 'val_mape_c':   va[5],
            'test_mae_r':   te[0], 'test_mae_c':   te[1],
            'test_rmse_r':  te[2], 'test_rmse_c':  te[3],
            'test_mape_r':  te[4], 'test_mape_c':  te[5],
        })
        if val_mape_avg < best_val_mape_avg:
            best_val_mape_avg = val_mape_avg
            mlp_model        = model
            best_mlp_params   = {'hidden_layer_sizes': hidden, 'alpha': alpha}
            train_time_mlp    = t_elapsed

mlp_experiments_df = pd.DataFrame(mlp_results)
print("MLP — hyperparameter tuning")
print(f"Data: n_train={n_train}, n_val={n_val}, n_test={n_test}")
print("Selection criterion: lowest avg validation MAPE across R and C.")
display(_fmt_df(mlp_experiments_df))
print(f"\nBest config (by validation MAPE): {best_mlp_params}")
print("Justification: Chosen hidden_layer_sizes and alpha minimize average validation MAPE across R and C.")
print("\nBest model — Val and Test metrics:")
y_val_pred  = to_phys(mlp_model.predict(X_val),  target_is_log)
y_test_pred = to_phys(mlp_model.predict(X_test), target_is_log)
va = metrics_r_c(y_val_phys,  y_val_pred)
te = metrics_r_c(y_test_phys, y_test_pred)
print(f"  Train time: {train_time_mlp:.4f} s")
print(f"  Val:  R mae: {va[0]:.4f}  C mae: {va[1]:.4e}  R rmse: {va[2]:.4f}  C rmse: {va[3]:.4e}  R mape: {va[4]:.2f}%  C mape: {va[5]:.2f}%")
print(f"  Test: R mae: {te[0]:.4f}  C mae: {te[1]:.4e}  R rmse: {te[2]:.4f}  C rmse: {te[3]:.4e}  R mape: {te[4]:.2f}%  C mape: {te[5]:.2f}%")

MLP — hyperparameter tuning
Data: n_train=1400, n_val=300, n_test=300
Selection criterion: lowest avg validation MAPE across R and C.


,hidden_layer_sizes,alpha,train_time_s,train_mae_r,train_mae_c,train_rmse_r,train_rmse_c,train_mape_r,train_mape_c,val_mae_r,val_mae_c,val_rmse_r,val_rmse_c,val_mape_r,val_mape_c,test_mae_r,test_mae_c,test_rmse_r,test_rmse_c,test_mape_r,test_mape_c
0,"(64, 32)",0.000100,2.4731,65.2798,1.1885e-07,105.6359,2.1931e-07,4.77%,13.14%,60.9572,1.2730e-07,91.1463,2.6258e-07,4.71%,14.19%,65.0828,1.3527e-07,101.6611,2.6491e-07,4.87%,12.39%
1,"(64, 32)",0.001000,3.2128,55.3146,8.1917e-08,86.6560,1.8513e-07,4.43%,10.37%,51.1234,8.5525e-08,74.7219,2.5044e-07,4.26%,10.51%,55.7154,1.0507e-07,84.7682,2.7969e-07,4.46%,9.79%
2,"(64, 32)",0.010000,3.9377,57.3601,1.1900e-07,93.8268,2.2565e-07,4.84%,11.74%,51.5096,1.2242e-07,80.0373,2.7391e-07,4.49%,12.16%,56.3780,1.5689e-07,90.9629,3.4697e-07,5.08%,12.34%
3,"(128, 64)",0.000100,8.7504,31.6211,1.0051e-07,48.8374,2.1077e-07,2.57%,11.98%,30.8602,1.1099e-07,45.8105,2.6010e-07,2.57%,12.47%,30.6349,1.2720e-07,46.7943,3.1788e-07,2.52%,11.61%
4,"(128, 64)",0.001000,6.6840,40.6135,9.9927e-08,60.0016,2.2923e-07,4.04%,8.92%,37.8403,1.0050e-07,52.3972,2.8645e-07,3.84%,9.12%,36.5420,1.2590e-07,52.6617,3.2060e-07,3.80%,9.41%
5,"(128, 64)",0.010000,7.7776,36.5153,8.9103e-08,56.7067,1.9377e-07,3.05%,7.95%,34.6906,9.9464e-08,52.4932,2.5955e-07,3.00%,8.91%,35.4619,1.2162e-07,52.8494,2.8877e-07,2.99%,8.62%
6,"(256, 128, 64)",0.000100,14.6646,36.1996,1.0462e-07,55.7377,2.5601e-07,3.11%,11.46%,34.9986,1.0738e-07,52.1493,2.8089e-07,2.97%,11.95%,32.2027,1.3764e-07,48.5303,3.2519e-07,2.66%,11.00%
7,"(256, 128, 64)",0.001000,5.8314,42.4192,1.4534e-07,66.0704,2.8345e-07,3.17%,13.57%,38.6817,1.3066e-07,59.6640,2.8863e-07,2.96%,13.65%,39.6468,1.8687e-07,59.9109,4.2165e-07,3.23%,13.88%
8,"(256, 128, 64)",0.010000,6.4972,48.9210,1.1058e-07,80.1214,3.0973e-07,4.12%,11.18%,45.9065,1.0132e-07,71.2452,2.9551e-07,3.98%,10.72%,45.0515,1.6273e-07,72.9461,4.9988e-07,3.79%,11.86%



Best config (by validation MAPE): {'hidden_layer_sizes': (128, 64), 'alpha': 0.01}
Justification: Chosen hidden_layer_sizes and alpha minimize average validation MAPE across R and C.

Best model — Val and Test metrics:
  Train time: 7.7776 s
  Val:  R mae: 34.6906  C mae: 9.9464e-08  R rmse: 52.4932  C rmse: 2.5955e-07  R mape: 3.00%  C mape: 8.91%
  Test: R mae: 35.4619  C mae: 1.2162e-07  R rmse: 52.8494  C rmse: 2.8877e-07  R mape: 2.99%  C mape: 8.62%


## Model comparison

The table below compares the selected configuration of each model on the **test set**. We report **MAE**, **RMSE**, and **MAPE** for **R** and **C** (in physical units). The **Time (s)** column reports total time as **training time + test-set inference time**.


In [31]:
target_is_log = globals().get('target_is_log', False)
y_test_phys = np.exp(y_test) if target_is_log else y_test

def to_phys(y_pred, is_log):
    return np.exp(y_pred) if is_log else y_pred

def mape_r_c(y_true, y_pred, eps=1e-12):
    denom_r = np.maximum(np.abs(y_true[:, 0]), eps)
    denom_c = np.maximum(np.abs(y_true[:, 1]), eps)
    return (np.mean(np.abs(y_true[:, 0] - y_pred[:, 0]) / denom_r) * 100,
            np.mean(np.abs(y_true[:, 1] - y_pred[:, 1]) / denom_c) * 100)

def metrics_r_c(y_true, y_pred):
    mae_r  = mean_absolute_error(y_true[:, 0], y_pred[:, 0])
    mae_c  = mean_absolute_error(y_true[:, 1], y_pred[:, 1])
    rmse_r = np.sqrt(mean_squared_error(y_true[:, 0], y_pred[:, 0]))
    rmse_c = np.sqrt(mean_squared_error(y_true[:, 1], y_pred[:, 1]))
    mr, mc = mape_r_c(y_true, y_pred)
    return mae_r, mae_c, rmse_r, rmse_c, mr, mc

# Measure test-set inference time for each best model
t0 = time.perf_counter(); lr_test_pred  = to_phys(lr_model.predict(X_test),   target_is_log); infer_time_lr  = time.perf_counter() - t0
t0 = time.perf_counter(); rf_test_pred  = to_phys(best_model.predict(X_test), target_is_log); infer_time_rf  = time.perf_counter() - t0
t0 = time.perf_counter(); mlp_test_pred = to_phys(mlp_model.predict(X_test),  target_is_log); infer_time_mlp = time.perf_counter() - t0

lr_te  = metrics_r_c(y_test_phys, lr_test_pred)
rf_te  = metrics_r_c(y_test_phys, rf_test_pred)
mlp_te = metrics_r_c(y_test_phys, mlp_test_pred)

total_time_lr  = train_time_lr  + infer_time_lr
total_time_rf  = train_time_rf  + infer_time_rf
total_time_mlp = train_time_mlp + infer_time_mlp

rows = [
    {"Method": "Linear Regression (Ridge)",
     "MAE (R)":    lr_te[0],  "MAE (C)":    lr_te[1],
     "RMSE (R)":   lr_te[2],  "RMSE (C)":   lr_te[3],
     "MAPE (R) %": lr_te[4],  "MAPE (C) %": lr_te[5],
     "Time (s)":   total_time_lr},
    {"Method": "Random Forest",
     "MAE (R)":    rf_te[0],  "MAE (C)":    rf_te[1],
     "RMSE (R)":   rf_te[2],  "RMSE (C)":   rf_te[3],
     "MAPE (R) %": rf_te[4],  "MAPE (C) %": rf_te[5],
     "Time (s)":   total_time_rf},
    {"Method": "MLP",
     "MAE (R)":    mlp_te[0], "MAE (C)":    mlp_te[1],
     "RMSE (R)":   mlp_te[2], "RMSE (C)":   mlp_te[3],
     "MAPE (R) %": mlp_te[4], "MAPE (C) %": mlp_te[5],
     "Time (s)":   total_time_mlp},
]
comparison_df = pd.DataFrame(rows)
fmt = {
    "MAE (R)":    "{:.4f}",
    "MAE (C)":    "{:.4e}",
    "RMSE (R)":   "{:.4f}",
    "RMSE (C)":   "{:.4e}",
    "MAPE (R) %": "{:.2f}%",
    "MAPE (C) %": "{:.2f}%",
    "Time (s)":   "{:.4f}",
}
display(comparison_df.style.format(fmt).hide(axis="index"))

Method,MAE (R),MAE (C),RMSE (R),RMSE (C),MAPE (R) %,MAPE (C) %,Time (s)
Linear Regression (Ridge),94.1966,1.4024e-07,137.9576,3.5513e-07,8.74%,12.49%,0.1185
Random Forest,20.9666,8.6104e-08,31.9406,2.8657e-07,2.73%,8.57%,185.6053
MLP,35.4619,1.2162e-07,52.8494,2.8877e-07,2.99%,8.62%,7.7799


## Best hyperparameters per model

The table below summarises the best hyperparameter configuration found for each model (chosen by lowest average validation MAPE across R and C), along with data quantities and training time.

In [32]:
n_train, n_val, n_test = len(X_train), len(X_val), len(X_test)
data_str = f"({n_train}, {n_val}, {n_test})"

summary_rows = [
    {
        "Model": "Linear Regression (Ridge)",
        "Best hyperparameters": f"alpha={best_lr_params.get('alpha', '—')}",
        "Data (n_train, n_val, n_test)": data_str,
        "Train time (s)": train_time_lr,
    },
    {
        "Model": "Random Forest",
        "Best hyperparameters": (
            f"n_estimators={best_rf_params.get('n_estimators', '—')}, "
            f"max_depth={best_rf_params.get('max_depth', '—')}"
        ),
        "Data (n_train, n_val, n_test)": data_str,
        "Train time (s)": train_time_rf,
    },
    {
        "Model": "MLP",
        "Best hyperparameters": (
            f"hidden_layer_sizes={best_mlp_params.get('hidden_layer_sizes', '—')}, "
            f"alpha={best_mlp_params.get('alpha', '—')}"
        ),
        "Data (n_train, n_val, n_test)": data_str,
        "Train time (s)": train_time_mlp,
    },
]

summary_df = pd.DataFrame(summary_rows)
display(summary_df.style.format({"Train time (s)": "{:.4f}"}).hide(axis="index"))


Model,Best hyperparameters,"Data (n_train, n_val, n_test)",Train time (s)
Linear Regression (Ridge),alpha=1.0,"(1400, 300, 300)",0.1180
Random Forest,"n_estimators=200, max_depth=10","(1400, 300, 300)",185.5848
MLP,"hidden_layer_sizes=(128, 64), alpha=0.01","(1400, 300, 300)",7.7776


## Hybrid ML + Gauss-Newton

### How the models are combined

The best ML model (Random Forest) is used as a **warm-start initializer** for the Gauss-Newton (GN) parameter estimator from `CircuitSimulator`:

1. **Step 1 - ML prediction:** Random Forest predicts initial estimates of R and C directly from the flattened waveform features. This is instantaneous and requires no knowledge of the circuit equations.
2. **Step 2 - Gauss-Newton refinement:** The predicted (R, C) are passed as the starting point to `CircuitSimulator.GaussNewton`, which minimizes the residual between the circuit-simulated waveform and the actual observed waveform. At each iteration it:
   - Runs a full Backward Euler simulation with the current (R, C) guess.
   - Computes the sensitivity (Jacobian) of each simulated signal with respect to R and C.
   - Updates (R, C) via the multiplicative GN step: beta = (J^T J)^-1 J^T r, then R *= exp(beta[0]), C *= exp(beta[1]).

### Why this combination improves accuracy

| Problem with ML alone | How Gauss-Newton corrects it |
|---|---|
| Predictions are unconstrained - ML may output physically inconsistent (R, C) | GN enforces physics: the final answer must be consistent with the circuit ODE |
| ML generalises from training data; waveform shapes outside the training distribution degrade accuracy | GN minimises waveform residuals directly, so it adapts to each individual sample |
| ML has systematic bias from limited training data | GN iteratively reduces residuals toward zero |
| Gauss-Newton alone is sensitive to initialisation and can diverge | ML provides a close warm start, making GN converge faster and reliably |

### Trade-offs

- **Computation:** Much slower than ML alone - each GN iteration requires a full Backward Euler simulation per sample.
- **Dependencies:** Requires the physics simulator and physical-unit waveforms (the dataset is stored standardized, so unstandardization must be applied before calling GN).
- **Best use case:** High-accuracy post-processing of a small number of predictions where accuracy matters more than throughput.


In [ ]:
%reload_ext autoreload
%autoreload 2
import importlib
import group_5_circuit_simulator
importlib.reload(group_5_circuit_simulator)
from group_5_circuit_simulator import CircuitSimulator

# Circuit simulation constants (matching dataset generation in test.py)
AMPLITUDE = 5.0
FREQUENCY = 60.0
DELTA_T   = 1e-4
N_STEPS   = 500
# Tiny epsilon avoids an off-by-one due to float rounding in BEuler's while(t < T)
T_SUB     = N_STEPS * DELTA_T - 1e-12

def unstandardize(waveform_std, mu, sigma):
    """Convert standardized (T,4) waveform back to physical units."""
    if mu is None or sigma is None:
        return waveform_std
    return waveform_std * sigma[0] + mu[0]

# If the load-data cell wasn't rerun, pull waveforms/scaling directly from the pickle
try:
    X_test_ts
except NameError:
    import pickle
    with open('group_5_dataset.pkl', 'rb') as f:
        _data_gn = pickle.load(f)
    X_test_ts = _data_gn['X_test']
    mu = _data_gn.get('mu', None)
    sigma = _data_gn.get('sigma', None)

target_is_log = globals().get('target_is_log', False)
sample_indices = [0, 1, 2, 3, 4]

# RF predictions (physical units) for the chosen test samples
rf_init = to_phys(best_model.predict(X_test[sample_indices]), target_is_log)
y_true_phys = np.exp(y_test[sample_indices]) if target_is_log else y_test[sample_indices]

hybrid_rows = []
for j, idx in enumerate(sample_indices):
    R_true, C_true = y_true_phys[j, 0], y_true_phys[j, 1]
    R_ml,   C_ml   = rf_init[j, 0],     rf_init[j, 1]

    # Unstandardize waveform back to physical units, then truncate to N_STEPS
    waveform = unstandardize(X_test_ts[idx], mu, sigma)[:N_STEPS]

    # Run Gauss-Newton starting from ML prediction
    mna = CircuitSimulator(AMPLITUDE, FREQUENCY, R_ml, C_ml)
    R_hybrid, C_hybrid, _ = mna.GaussNewton(
        R_ml, C_ml, np.zeros(4), waveform, DELTA_T, T_SUB, max_iter=10
    )

    hybrid_rows.append({
        "idx":            idx,
        "R_true":         R_true,
        "C_true":         C_true,
        "R_ml":           R_ml,
        "C_ml":           C_ml,
        "R_hybrid":       R_hybrid,
        "C_hybrid":       C_hybrid,
        "|err_R| ML":     abs(R_ml     - R_true),
        "|err_R| Hybrid": abs(R_hybrid - R_true),
        "|err_C| ML":     abs(C_ml     - C_true),
        "|err_C| Hybrid": abs(C_hybrid - C_true),
    })

hybrid_df = pd.DataFrame(hybrid_rows).set_index("idx")
fmt_h = {
    "R_true": "{:.2f}", "R_ml": "{:.2f}", "R_hybrid": "{:.2f}",
    "|err_R| ML": "{:.4f}", "|err_R| Hybrid": "{:.4f}",
    "C_true": "{:.4e}", "C_ml": "{:.4e}", "C_hybrid": "{:.4e}",
    "|err_C| ML": "{:.4e}", "|err_C| Hybrid": "{:.4e}",
}
print(f"Hybrid ML + Gauss-Newton: parameter estimation on {len(sample_indices)} test samples")
print(f"(GN uses first {N_STEPS} time steps, max_iter=10, warm-started from Random Forest)")
display(hybrid_df.style.format(fmt_h))

mae_r_ml     = hybrid_df["|err_R| ML"].mean()
mae_r_hybrid = hybrid_df["|err_R| Hybrid"].mean()
mae_c_ml     = hybrid_df["|err_C| ML"].mean()
mae_c_hybrid = hybrid_df["|err_C| Hybrid"].mean()
print(f"\nMean |err R|: ML = {mae_r_ml:.4f} ohm  -->  Hybrid = {mae_r_hybrid:.4f} ohm  "
      f"({'improved' if mae_r_hybrid < mae_r_ml else 'no improvement'})")
print(f"Mean |err C|: ML = {mae_c_ml:.4e} F    -->  Hybrid = {mae_c_hybrid:.4e} F    "
      f"({'improved' if mae_c_hybrid < mae_c_ml else 'no improvement'})")


Number of iterations: 10.
Number of iterations: 10.
Number of iterations: 10.
Number of iterations: 10.
Number of iterations: 10.
Hybrid ML + Gauss-Newton: parameter estimation on 5 test samples
(GN uses first 500 time steps, max_iter=10, warm-started from Random Forest)


,R_true,C_true,R_ml,C_ml,R_hybrid,C_hybrid,|err_R| ML,|err_R| Hybrid,|err_C| ML,|err_C| Hybrid
idx,,,,,,,,,,
0,1350.59,3.2323e-07,1342.78,2.8908e-07,1429.54,3.0590e-07,7.8109,78.9475,3.4155e-08,1.7334e-08
1,2204.25,1.5522e-07,2181.78,1.4779e-07,2112.45,1.6918e-07,22.4728,91.7989,7.4291e-09,1.3955e-08
2,2406.40,1.5091e-06,2360.61,1.5059e-06,2393.51,1.5159e-06,45.7903,12.8839,3.2088e-09,6.7850e-09
3,1114.44,3.3354e-06,1129.87,3.3980e-06,1113.71,3.3438e-06,15.4304,0.7308,6.2569e-08,8.3512e-09
4,361.35,2.3552e-07,344.01,2.1411e-07,357.76,1.6825e-07,17.3346,3.5893,2.1409e-08,6.7276e-08



Mean |err R|: ML = 21.7678 ohm  -->  Hybrid = 37.5901 ohm  (no improvement)
Mean |err C|: ML = 2.5754e-08 F    -->  Hybrid = 2.2740e-08 F    (improved)
